# Task 3: Sentiment Analysis and Correlation with Stock Returns
 Quantify the relationship between financial news sentiment and daily stock price movements.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from textblob import TextBlob
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
print("Libraries loaded")

Libraries loaded


In [2]:
import sys
print(sys.executable)

c:\Users\Bitan\news-sentiment-analysis-1\venv\Scripts\python.exe


In [3]:
import sys
!{sys.executable} -m pip install textblob
!{sys.executable} -m textblob.download_corpora

Finished.


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\Bitan\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Bitan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Bitan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Bitan\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to
[nltk_data]     C:\Users\Bitan\AppData\Roaming\nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\Bitan\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is alr

In [5]:
import os
os.chdir('C:\\Users\\Bitan\\news-sentiment-analysis-1')
print("Current working directory:", os.getcwd())

Current working directory: C:\Users\Bitan\news-sentiment-analysis-1


In [6]:
df_news = pd.read_csv('data/raw/raw_analyst_ratings.csv', parse_dates=['date'])

if 'Unnamed: 0' in df_news.columns:
    df_news.drop(columns=['Unnamed: 0'], inplace=True)

df_news['stock'] = df_news['stock'].str.upper()
df_news = df_news.dropna(subset=['headline', 'stock'])

print(f"News shape: {df_news.shape}")
df_news.head()

News shape: (1407328, 5)


,headline,url,publisher,date,stock
0,Stocks That Hit 52-Week Highs On Friday,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,2020-06-05 10:30:54-04:00,A
1,Stocks That Hit 52-Week Highs On Wednesday,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,2020-06-03 10:45:20-04:00,A
2,71 Biggest Movers From Friday,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,2020-05-26 04:30:07-04:00,A
3,46 Stocks Moving In Friday's Mid-Day Session,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,2020-05-22 12:45:06-04:00,A
4,B of A Securities Maintains Neutral on Agilent...,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,2020-05-22 11:38:59-04:00,A


In [7]:
top_stocks = ['MRK', 'MS', 'NVDA', 'MU', 'QQQ']

df_news = df_news[df_news['stock'].isin(top_stocks)]
print(f"Filtered shape: {df_news.shape}")
df_news['stock'].value_counts()

Filtered shape: (15965, 5)


stock
MRK     3333
MS      3238
NVDA    3146
MU      3142
QQQ     3106
Name: count, dtype: int64

In [8]:
prices = yf.download(top_stocks, start='2009-01-01', end='2020-07-01', group_by='ticker')
print(prices.head())

[*********************100%***********************]  5 of 5 completed

Ticker             MS                                                   QQQ  \
Price            Open       High        Low      Close    Volume       Open   
Date                                                                          
2009-01-02  10.977915  11.684177  10.799635  11.649892  20238000  25.679824   
2009-01-05  11.519611  12.589290  11.505898  12.081879  25421400  26.612079   
2009-01-06  12.390436  13.665822  12.342438  13.425830  38858500  26.974610   
2009-01-07  13.076127  13.288691  12.177872  12.411007  30848100  26.525747   
2009-01-08  12.273868  13.014415  12.102446  12.904704  21116300  26.223622   

Ticker                                                  ...        MRK  \
Price            High        Low      Close     Volume  ...       Open   
Date                                                    ...              
2009-01-02  26.871022  25.628033  26.784704  107891500  ...  15.975297   
2009-01-05  27.043673  26.404914  26.776085   91751900  ...  16.494521 

In [12]:
# --- Step 1: Convert news dates to UTC, then naive ---
df_news['date'] = pd.to_datetime(df_news['date'], utc=True, errors='coerce')
df_news['date'] = df_news['date'].dt.tz_localize(None)   # remove timezone
df_news = df_news.dropna(subset=['date'])                # drop bad dates

# --- Step 2: Prepare trading days (already naive) ---
trading_days = prices[top_stocks[0]].index   # from yfinance download

# --- Step 3: Align news to next trading day ---
def next_trading_day(date):
    future = trading_days[trading_days >= date]
    return future[0] if len(future) > 0 else None

df_news['trading_date'] = df_news['date'].apply(next_trading_day)
df_news = df_news.dropna(subset=['trading_date'])
print("Alignment done. Rows with valid trading date:", len(df_news))
df_news.head()

Alignment done. Rows with valid trading date: 50


,headline,url,publisher,date,stock,trading_date
846406,Shares of several healthcare companies are tra...,https://www.benzinga.com/wiim/20/06/16233278/s...,Benzinga Newsdesk,2020-06-11 14:22:31,MRK,2020-06-12
846407,Johnson & Johnson To Start Coronavirus Vaccine...,https://www.benzinga.com/general/biotech/20/06...,Neer Varshney,2020-06-11 04:16:21,MRK,2020-06-12
846408,The Daily Biotech Pulse: Keytruda Setback For ...,https://www.benzinga.com/general/biotech/20/06...,Shanthi Rexaline,2020-06-10 11:30:59,MRK,2020-06-11
846409,Merck Announces That The Phase 3 KEYNOTE-361 T...,https://www.benzinga.com/news/20/06/16216257/m...,Benzinga Newsdesk,2020-06-09 20:13:02,MRK,2020-06-10
846410,"The Week Ahead In Biotech: Viela FDA Decision,...",https://www.benzinga.com/general/biotech/20/06...,Shanthi Rexaline,2020-06-07 17:43:52,MRK,2020-06-08


In [13]:
# Make news dates timezone-naive for comparison
df_news['date_naive'] = df_news['date'].dt.tz_localize(None)

# Get all trading days from price data (already naive)
trading_days = prices[top_stocks[0]].index

def next_trading_day(date):
    future = trading_days[trading_days >= date]
    return future[0] if len(future) > 0 else None

df_news['trading_date'] = df_news['date_naive'].apply(next_trading_day)
df_news = df_news.dropna(subset=['trading_date'])
df_news.head()

,headline,url,publisher,date,stock,trading_date,date_naive
846406,Shares of several healthcare companies are tra...,https://www.benzinga.com/wiim/20/06/16233278/s...,Benzinga Newsdesk,2020-06-11 14:22:31,MRK,2020-06-12,2020-06-11 14:22:31
846407,Johnson & Johnson To Start Coronavirus Vaccine...,https://www.benzinga.com/general/biotech/20/06...,Neer Varshney,2020-06-11 04:16:21,MRK,2020-06-12,2020-06-11 04:16:21
846408,The Daily Biotech Pulse: Keytruda Setback For ...,https://www.benzinga.com/general/biotech/20/06...,Shanthi Rexaline,2020-06-10 11:30:59,MRK,2020-06-11,2020-06-10 11:30:59
846409,Merck Announces That The Phase 3 KEYNOTE-361 T...,https://www.benzinga.com/news/20/06/16216257/m...,Benzinga Newsdesk,2020-06-09 20:13:02,MRK,2020-06-10,2020-06-09 20:13:02
846410,"The Week Ahead In Biotech: Viela FDA Decision,...",https://www.benzinga.com/general/biotech/20/06...,Shanthi Rexaline,2020-06-07 17:43:52,MRK,2020-06-08,2020-06-07 17:43:52


In [14]:
def get_sentiment(text):
    return TextBlob(str(text)).sentiment.polarity

df_news['sentiment'] = df_news['headline'].apply(get_sentiment)
df_news[['headline', 'sentiment']].head()

,headline,sentiment
846406,Shares of several healthcare companies are tra...,0.0
846407,Johnson & Johnson To Start Coronavirus Vaccine...,0.0
846408,The Daily Biotech Pulse: Keytruda Setback For ...,0.0
846409,Merck Announces That The Phase 3 KEYNOTE-361 T...,0.4
846410,"The Week Ahead In Biotech: Viela FDA Decision,...",-0.1


In [15]:
daily_sentiment = df_news.groupby(['trading_date', 'stock'])['sentiment'].mean().reset_index()
daily_sentiment.head()

,trading_date,stock,sentiment
0,2020-05-29,MU,0.00
1,2020-06-01,MU,0.00
2,2020-06-01,NVDA,0.00
3,2020-06-02,MU,0.10
4,2020-06-03,NVDA,0.25


In [ ]:
all_returns = []
for ticker in top_stocks:
    temp = prices[ticker].copy()
    if 'Adj Close' in temp.columns:
        price_col = 'Adj Close'
    elif 'Adj Close' in temp.columns.get_level_values(0):
        temp.columns = temp.columns.droplevel(0)
        price_col = 'Adj Close'
    else:
        price_col = 'Close'
    
    temp = temp[[price_col]].rename(columns={price_col: 'close'})
    temp['return'] = temp['close'].pct_change() * 100
    temp['stock'] = ticker
    temp = temp.reset_index().rename(columns={'index': 'trading_date'})
    all_returns.append(temp)

df_returns = pd.concat(all_returns, ignore_index=True)
df_returns.head()

Price,Date,close,return,stock
0,2009-01-02,16.258511,NaN,MRK
1,2009-01-05,16.012011,-1.516129,MRK
2,2009-01-06,15.718313,-1.834232,MRK
3,2009-01-07,15.466556,-1.601683,MRK
4,2009-01-08,15.398383,-0.440773,MRK


In [22]:
df_returns.rename(columns={'Date': 'trading_date'}, inplace=True)

merged = daily_sentiment.merge(df_returns, on=['trading_date', 'stock'], how='inner')
merged = merged.dropna()
merged.head()

,trading_date,stock,sentiment,close,return
0,2020-05-29,MU,0.00,46.726006,3.098769
1,2020-06-01,MU,0.00,45.194809,-3.276969
2,2020-06-01,NVDA,0.00,8.771857,-0.780224
3,2020-06-02,MU,0.10,45.662952,1.035835
4,2020-06-03,NVDA,0.25,8.735250,-0.631712


In [32]:
df_news_full = pd.read_csv('data/raw/raw_analyst_ratings.csv', parse_dates=['date'])
if 'Unnamed: 0' in df_news_full.columns:
    df_news_full.drop(columns=['Unnamed: 0'], inplace=True)

# Use mixed format to parse all dates, then convert to UTC, then naive
df_news_full['date'] = pd.to_datetime(df_news_full['date'], format='mixed', utc=True, errors='coerce')
df_news_full['date'] = df_news_full['date'].dt.tz_localize(None)
df_news_full = df_news_full.dropna(subset=['headline', 'date'])